# Лабораторная 2 · Каскад: разведчик ищет, судья решает

**Неделя 7 · занятие 1.** Опора — L10 «Разведчики и судьи»: би-энкодеры, кросс-энкодеры,
многоступенчатые пайплайны.

Кросс-энкодер точнее би-энкодера и в тысячи раз дороже. Из этого следует вся конструкция
современного поиска — и сегодня ты соберёшь её руками и померяешь, **сколько она на самом деле
даёт**. Ответ будет неприятным и полезным.

| # | вопрос занятия | чем отвечаем |
|---|---|---|
| 1 | Почему нельзя обойтись одной ступенью? | считаем задержку обеих на всём корпусе |
| 2 | Чем занят би-энкодер, если он не точнее BM25? | меряем MRR и Recall отдельно — они расходятся |
| 3 | На какую глубину переранжировать? | прогон по глубине, парный тест, цена в миллисекундах |

**Данные.** 20 Newsgroups, шесть категорий, 1500 документов — де-факто свободный корпус
из поставки scikit-learn. Запросы **псевдо**: предложение, взятое из документа и удалённое
из индексируемого текста; релевантен ровно один документ. Числа MS MARCO из
`data/l7-msmarco.json` мы **цитируем**, но не используем как датасет: лицензия некоммерческая,
а курс платный.

**Среда.** Colab T4 через VS Code. Модели маленькие (22M и 22M параметров) и идут на CPU
за пару минут — GPU ускорит, но не обязателен.

**Бюджет: ≈120 минут.**

**Артефакт на вынос.** `artifacts/cascade.json` — эмбеддинги корпуса, выдачи обеих ступеней
и все замеры. На неделе 9 гибридное ранжирование возьмёт их как две из трёх ступеней.

**Как запускать.** Сверху вниз. Корпус сегодня **свой** — шесть категорий вместо двух и другие
запросы, поэтому индекс недели 3 не подходит и строится заново. А вот `metrics.py` с недели 4
подхватывается, и ноутбук проверяет, что его реализации согласны с нашими: две разные реализации
одной метрики — это способ померить собственную опечатку вместо эффекта.

<details><summary>Почему две ступени, а не одна модель получше</summary>

Потому что вычислительная стоимость двух подходов различается не в разы, а на порядки, и это
не инженерная деталь, а следствие устройства.

**Би-энкодер** кодирует запрос и документ **независимо**. Документы кодируются один раз,
заранее, и складываются в индекс. На запросе остаётся закодировать сам запрос и найти ближайшие
векторы — это одно матричное умножение либо обращение к ANN-индексу. Стоимость на запрос
не зависит от размера корпуса (при ANN — почти не зависит).

**Кросс-энкодер** кодирует пару «запрос-документ» **вместе**: токены запроса и документа
попадают в один вход, и внимание работает между ними. Именно поэтому он точнее — он видит
взаимодействие слов запроса со словами документа напрямую, а не через два сжатых вектора.
И именно поэтому его нельзя предвычислить: пары не существует, пока не пришёл запрос.

**Арифметика.** Миллион документов, кросс-энкодер 12 мс на пару. Один запрос = 12 000 секунд,
то есть три с половиной часа. С батчами и GPU — минуты, всё равно неприемлемо. Отсюда
единственная работающая конструкция: дешёвая ступень отбирает сотню кандидатов, дорогая
переупорядочивает сотню.

**Что отсюда следует для качества.** Потолок каскада задаёт **первая** ступень: то, чего она
не вернула, вторая уже не найдёт. Поэтому от первой ступени требуют полноты, а не точности,
и меряют её Recall@k, а не MRR. Мы это увидим в числах через две части.
</details>

<details><summary>Три архитектуры между би- и кросс-энкодером, и почему они существуют</summary>

Между «сжать документ в один вектор заранее» и «считать пару целиком на запросе» лежит
пространство компромиссов, и все они пытаются купить точность, не заплатив предвычислимостью.

**Late interaction (ColBERT).** Документ кодируется заранее, но не в один вектор, а в вектор
**на каждый токен**. На запросе для каждого токена запроса ищется максимально похожий токен
документа, и скоры складываются. Взаимодействие есть, но оно отложено на самый конец и потому
дёшево. Цена — память: вместо 384 чисел на документ хранится 384 × число токенов, то есть
в сотню раз больше. Об этом лекция L12 и семинар недели 9.

**Разреженные обучаемые представления (SPLADE).** Документ кодируется в вектор размерности
всего словаря, где почти все координаты нули, а ненулевые — это термины с обучёнными весами,
включая термины, которых в документе нет. Получается что-то среднее между BM25 и плотным
поиском: инвертированный индекс работает как обычно, а веса приходят от модели. Предвычислимо
полностью.

**Дистилляция кросс-энкодера в би-энкодер.** Кросс-энкодер размечает много пар, би-энкодер
учится воспроизводить его скоры. Точность частично переносится, стоимость остаётся
би-энкодерной. Это самый практичный путь и то, чем реально занимаются в проде: почти все
сильные открытые ретриверы обучены дистилляцией с кросс-энкодера.

**Что объединяет все три.** Каждая пытается сохранить хотя бы часть взаимодействия
«запрос — документ», не откладывая всю работу на время запроса. Разница между ними —
в том, какой ресурс приносится в жертву: память, размер индекса или качество дистилляции.

**Почему мы не делаем этого сегодня.** Двухступенчатый каскад — минимальная конструкция,
на которой видно разделение труда. Добавлять третий вариант, не измерив первые два,
значило бы получить три числа и ни одного вывода.
</details>

## Шаг 0 · Пины и preflight

Список проверок вырос: добавились модели, которые надо скачать, и память, которой может
не хватить при неудачном батче.

In [ ]:
# ПИНЫ — точнее, ОТКАЗ от них там, где они ломают Colab.
# Базовый стек образа (numpy, scipy, scikit-learn, matplotlib, torch) собран сам под себя.
# Понижать его нельзя: `pip install numpy==1.26.4` откатывает ОДИН numpy, а scipy и sklearn
# остаются собранными под numpy 2 — и первый же импорт падает с
# «ModuleNotFoundError: No module named 'numpy.char'». Ставим ТОЛЬКО то, чего в образе нет.
import importlib.util as _ilu, subprocess as _sp, sys as _sys

NEEDED = {"sentence_transformers": "sentence-transformers"}
_missing = [pkg for mod, pkg in NEEDED.items() if _ilu.find_spec(mod) is None]
if _missing:
    print("ставлю:", ", ".join(_missing))
    _sp.run([_sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    print("готово · если следующий импорт упадёт — Runtime → Restart session, потом эта ячейка снова")
else:
    print("всё нужное уже в образе Colab — ставить нечего")

import json, math, os, random, re, statistics, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer, CrossEncoder

In [ ]:
def preflight():
    problems = []
    try:
        fetch_20newsgroups(subset="train", categories=["sci.space"], download_if_missing=False)
    except Exception:
        problems.append("20NG не в кэше: первый вызов тянет ~14 МБ. В Colab это норма.")
    if not Path(f"{DATA_DIR}/l7-biencoder.json").exists():
        problems.append(f"нет {DATA_DIR}/l7-*.json -- сверка с лекцией невозможна.")
    if not METRICS_PATH.exists():
        problems.append(
            f"нет {METRICS_PATH} (metrics.py с недели 4). Это НЕ ошибка: ноутбук определит "
            "MRR и Recall у себя. Но это ВТОРАЯ реализация тех же метрик, и если она разойдётся "
            "с первой, ты будешь мерить свою опечатку.")
    if not torch.cuda.is_available():
        print("· GPU нет -- идём на CPU. Кодирование корпуса ~1 мин, свип глубины ~2 мин.")
    for p in problems:
        print("!", p)
    print("preflight:", "ЧИСТО" if not problems else f"{len(problems)} замечани(я/й) -- читай выше")
    return not problems

## Шаг 1 · Конфигурация

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

SMOKE = os.environ.get("SMOKE", "0") == "1"
CORPUS_N = 300 if SMOKE else 1500
N_QUERIES = 20 if SMOKE else 80
DEPTHS = (10, 25, 50, 100)
CATS = ["sci.space", "rec.sport.hockey", "comp.graphics",
        "talk.politics.mideast", "sci.med", "rec.autos"]
BI_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

DATA_DIR = os.environ.get("DLS_DATA", "./data")
ARTIFACTS = Path(os.environ.get("ARTIFACTS", "./artifacts"))
METRICS_PATH = ARTIFACTS / "metrics.py"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

RUN = {"seed": SEED, "smoke": SMOKE, "corpus_n": CORPUS_N, "n_queries": N_QUERIES,
       "depths": list(DEPTHS), "bi": BI_MODEL, "ce": CE_MODEL}
print(json.dumps(RUN, ensure_ascii=False, indent=1))
preflight()

**Что видно.** Конфигурация напечатана целиком, включая **имена моделей** — и это не
формальность: `all-MiniLM-L6-v2` и `ms-marco-MiniLM-L-6-v2` это две разные модели с разными
обучающими данными, и все числа ниже привязаны именно к ним. Сравнивать надо не строки вывода,
а `smoke` с тем, чего ты ждёшь: в быстром режиме корпус вчетверо меньше, а меньший корпус —
это более лёгкая задача поиска, и все метрики окажутся выше. Механизм прямой: при 300 документах
у правильного ответа меньше конкурентов. Чего этот вывод НЕ показывает: скачались ли модели —
это выяснится в части 2 и займёт минуту. Что делать: если гоняешь в `SMOKE`, не сравнивай свои
числа с числами соседа и с числами из разбора ниже.

---

## Часть 1 · Арифметика, из которой следует каскад — 20 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 1.1 | Сколько стоит кросс-энкодер на всём корпусе? | считаем по цитируемым числам лекции |
| 1.2 | Что даёт воронка? | считаем ту же цену при отборе сотни |
| 1.3 | Какая база у нас есть? | BM25 с недели 3 — наше нулевое число |

### Шаг 1.1 · Цена одной ступени

**Внешние числа.** Всё в этой ячейке взято из `data/l7-cascade.json` — это цитируемые
замеры задержки для представительных моделей, **не наши**. Наши собственные замеры появятся
в части 4 и будут помечены отдельно. Смешивать их в одной таблице нельзя.

In [ ]:
C = json.load(open(f"{DATA_DIR}/l7-cascade.json", encoding="utf-8"))
lat = C["latency"]

corpus_sizes = [1_000, 100_000, 1_000_000]
print(f"{'корпус':>10} {'кросс-энкодер на ВСЁМ':>24} {'каскад (отбор ' + str(lat['rerankDepth']) + ')':>22}")
for n in corpus_sizes:
    full_ms = n * lat["crossPerPairMs"]
    print(f"{n:>10,} {full_ms / 1000:>21,.0f} c {lat['totalMs']:>19} мс")
print()
print("состав каскада (цитируется, не наш замер):")
for k in ("queryEncodeMs", "annSearchMs", "rerankMs", "totalMs"):
    print(f"   {k:>16} = {lat[k]} мс")
print(f"   источник: {lat['source'][:80]}...")
RUN["cited_cascade_ms"] = lat["totalMs"]

**Что видно.** На миллионе документов кросс-энкодер в лоб стоит около трёх с половиной часов
на **один** запрос, а каскад — 133 миллисекунды. Сравнивать надо не два числа как «медленно
и быстро», а **порядки — и на одном основании**. В таблице основания смешаны, и источник сам
об этом предупреждает: колонка «в лоб» оценена по цене одиночной пары на CPU, каскад — по
батчевому замеру на GPU. Приведи обе к батчевому GPU — полный проход миллиона пар это
десятки минут против 133 мс, разница на **четыре порядка**, — и она всё равно не закрывается
ни железом, ни оптимизацией. Механизм в том, что стоимость первой колонки линейна по корпусу, а второй —
константна: глубина переранжирования не зависит от того, миллион у тебя документов или десять.
Чего эта таблица НЕ показывает: качества. Она показывает только, что одноступенчатый
кросс-энкодер невозможен, а не что каскад хорош. Что делать: держать в голове, что это
**чужие** числа с чужого железа. Свои мы померяем в части 4, и они будут другими.

<details><summary>Почему rerankMs не равно глубине, умноженной на цену пары</summary>

В цитируемых числах `crossPerPairMs = 12`, глубина 100, а `rerankMs = 120`, а не 1200.
Разница в десять раз — не опечатка, и понимать её нужно, иначе любая оценка стоимости каскада
окажется завышенной на порядок.

**Батч меняет арифметику.** Кросс-энкодер на ускорителе считает не одну пару, а тензор из
тридцати двух пар за один проход. Время прохода растёт с размером батча гораздо медленнее,
чем линейно, пока батч не упрётся в память или в вычислительную ёмкость. Цена «на пару»
поэтому величина производная: она равна времени батча, делённому на его размер, и падает
с ростом батча.

**Откуда тогда 12 мс.** Это цена **одиночной** пары — то есть худший случай, батч из одного
элемента, где вся стоимость запуска ядра ложится на одну пару. Такое число полезно для оценки
задержки одного вызова и бесполезно для оценки пропускной способности.

**Практическое следствие для проектирования.** Глубина переранжирования почти бесплатна,
пока помещается в один батч, и дорожает ступенькой, когда перестаёт помещаться. Глубина 32
и глубина 64 при батче 32 отличаются не вдвое, а ровно на один дополнительный проход. Именно
поэтому глубины выбирают кратными батчу, а не круглыми числами вроде 100.

**Что это значит для наших замеров.** Мы мерили на CPU с батчем 32, и наши миллисекунды
на глубине 10 (батч недозагружен) завышены относительно того, что было бы при полном батче.
То есть наша кривая цены **пессимистична на малых глубинах** — реальная экономия от малой
глубины ещё больше, чем показывают наши столбики.
</details>

⚠️ Ловушка F · **Цитируемая задержка не переносится.** 12 мс на пару — это конкретная модель
на конкретном ускорителе при конкретной длине текста. На CPU будет в разы медленнее, на длинных
документах — ещё медленнее, при батчинге — быстрее в пересчёте на пару. Заметь, что в данных
`rerankMs = 120` при `crossPerPairMs = 12` и глубине 100: это **не** 100 × 12, потому что сотня
пар считается батчем. Число, полученное умножением, было бы завышено в десять раз.

### Шаг 1.2 · Корпус и псевдозапросы

Нам нужен корпус с известными правильными ответами. Разметка людьми недоступна, категории
20NG слишком грубы — на прошлом занятии мы видели, к чему это приводит. Берём третий путь:
**псевдозапросы**. Из документа вынимается одно предложение, оно становится запросом,
а из индексируемого текста удаляется. Релевантен ровно один документ — тот, откуда предложение.

In [ ]:
raw = fetch_20newsgroups(subset="train", categories=CATS,
                         remove=("headers", "footers", "quotes"), random_state=SEED)
TOKEN = re.compile(r"[a-z]{2,}")
SENT = re.compile(r"(?<=[.!?])\s+")

pool = []
for t in raw.data:
    t = " ".join(t.split())
    if len(TOKEN.findall(t.lower())) < 80:
        continue
    sents = [s for s in SENT.split(t) if 12 <= len(s.split()) <= 25]
    if sents:
        pool.append((t, sents))
random.Random(SEED).shuffle(pool)

DOCS = [p[0] for p in pool[:CORPUS_N]]
QUERIES = []
for i in range(N_QUERIES):
    text, sents = pool[i]
    s = max(sents, key=len)
    QUERIES.append(s.strip())
    DOCS[i] = text.replace(s, "").strip()      # предложение удалено из индекса
GOLD = list(range(N_QUERIES))                   # релевантен ровно документ с тем же индексом

print(f"корпус: {len(DOCS)} документов · запросов: {len(QUERIES)} · релевантных на запрос: 1")
print(f"медианная длина документа: {int(np.median([len(d.split()) for d in DOCS]))} слов")
print(f"медианная длина запроса:   {int(np.median([len(q.split()) for q in QUERIES]))} слов")
print(f"\nпример запроса: {QUERIES[0][:110]}")

**Что видно.** Запросы длиннее реальных пользовательских в разы — это целые предложения
по восемнадцать слов, а не «нхл плейофф». Сравнивать надо не длины друг с другом, а **длину
запроса с типом задачи**: длинный запрос-предложение это скорее «найди похожий абзац»,
чем «найди ответ». Механизм такой из-за способа построения — мы взяли готовое предложение,
потому что размечать вручную нечем. Чего эта конструкция НЕ даёт: реалистичного распределения
запросов. Что делать: помнить об этом при чтении всех чисел ниже и не переносить их на продукт
с короткими запросами.

<details><summary>Псевдозапросы: откуда приём, чем он смещён и как делают лучше</summary>

Приём «взять предложение из документа и назвать его запросом» — не наша выдумка. Он называется
inverse cloze task и лежит в основе обучения ретриверов без разметки (ICT в работе Lee et al.
2019, из которой выросла половина современных плотных ретриверов).

**Почему он работает.** Предложение и остальной документ связаны темой, стилем и словарём.
Модель, учась находить документ по вырезанному из него предложению, учится измерять
тематическую близость, а это и есть нужный сигнал.

**Чем он смещён, по пунктам.**
Первое: словарь общий. Реальный пользователь напишет «сколько живёт кошка», а в документе будет
«продолжительность жизни домашних кошек составляет». Пересечения слов почти нет, а у нас оно
почти полное — отсюда завышенная сила BM25.
Второе: длина. Наши запросы по восемнадцать слов, реальные — по два-четыре. Длинный запрос
даёт больше сигнала обеим системам, но особенно лексической.
Третье: стиль. Запрос написан автором документа, а не пользователем, и наследует его лексику,
опечатки и жаргон.
Четвёртое: ровно один правильный ответ. В реальности их несколько, и это меняет поведение
всех метрик полноты.

**Как делают лучше при наличии бюджета.** Генерируют запросы моделью: берут документ, просят
LLM написать вопрос, на который он отвечает, и получают текст, стилистически похожий
на пользовательский. Это тоже прокси, но смещённый **иначе**, и потому полезно померять
на обоих: если выводы совпадают на псевдозапросах и на сгенерированных, доверия им больше.

**Чего не делает ни один прокси.** Не воспроизводит распределение реальных запросов — с длинным
хвостом, опечатками, навигационными и транзакционными намерениями. Это достижимо только логами.
</details>

⚠️ Ловушка A · **Псевдозапрос делится словарём с целевым документом.** Мы удалили само
предложение, но остальной текст написан тем же человеком, на ту же тему и теми же словами.
Лексическое пересечение поэтому завышено, и смещение это **в пользу BM25**: он находит документ
по словам, которые в реальном запросе пользователя не встретились бы. Все сравнения «нейросеть
против BM25» ниже занижают преимущество нейросети, а не завышают.

⚠️ Ловушка A · **Один релевантный документ на запрос — это упрощение.** В реальности на запрос
отвечают несколько документов, и метрики полноты ведут себя иначе. Наша конструкция делает
Recall@k равным доле запросов, где нужный документ вообще попал в топ-k, — это по сути
«успех/неуспех», а не полнота в полном смысле.

### Шаг 1.3 · Число, с которым сравнивается всё остальное

BM25 с недели 3. Он не «слабый бейзлайн для галочки» — на многих задачах он до сих пор
обыгрывает нейросети, и сегодня мы увидим, насколько близко он держится.

In [ ]:
def toks(s):
    return TOKEN.findall(s.lower())

post, doclen = defaultdict(dict), {}
for i, d in enumerate(DOCS):
    tk = toks(d)
    doclen[i] = len(tk)
    for t, tf in Counter(tk).items():
        post[t][i] = tf
N, AVGDL = len(DOCS), sum(doclen.values()) / len(DOCS)

def bm25_rank(query, k1=1.5, b=0.75):
    # -> (порядок, скоры в этом же порядке). Скоры нужны для слияния на неделе 9:
    # из порядка нельзя узнать, обошёл документ соседа на волос или на пропасть.
    sc = defaultdict(float)
    for t in toks(query):
        p = post.get(t)
        if not p:
            continue
        idf = math.log((N - len(p) + 0.5) / (len(p) + 0.5) + 1)
        for d, tf in p.items():
            sc[d] += idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doclen[d] / AVGDL))
    order = sorted(sc, key=lambda d: (-sc[d], d))
    return order, [sc[d] for d in order]

def rr_at(order, gold, k=10):
    for i, d in enumerate(order[:k], 1):
        if d == gold:
            return 1 / i
    return 0.0

def recall_at(order, gold, k):
    return 1.0 if gold in order[:k] else 0.0

t0 = time.perf_counter()
BM, BM_SCORES = zip(*(bm25_rank(q) for q in QUERIES))
BM, BM_SCORES = list(BM), list(BM_SCORES)
t_bm = (time.perf_counter() - t0) / len(QUERIES)

BASE = statistics.mean(rr_at(o, g) for o, g in zip(BM, GOLD))
print(f"BASE -- BM25 MRR@10 = {BASE:.4f}   ({t_bm * 1000:.1f} мс/запрос, CPU, без батча)")
for k in (10, 50, 100, 200):
    print(f"   BM25 Recall@{k:<4} = {statistics.mean(recall_at(o, g, k) for o, g in zip(BM, GOLD)):.4f}")
RUN["bm25_mrr10"], RUN["bm25_ms"] = BASE, t_bm * 1000

**Что видно.** BM25 находит нужный документ в первой десятке примерно в шести случаях
из десяти, а MRR@10 около 0,43. Раздели одно на другое: среди **найденных** средний обратный
ранг 0,43 / 0,62 ≈ 0,7 — найденный ответ обычно стоит первым или вторым. Вниз MRR тянет
не позиция, а те четыре запроса из десяти, где документа нет в десятке вовсе. Сравнивать надо не MRR с Recall, а **их поведение вместе**:
Recall@10 заметно выше MRR, значит документ часто попадает в десятку, но не на первое место.
Механизм для псевдозапросов ожидаемый: лексического пересечения хватает, чтобы попасть в топ,
и не хватает, чтобы отличить нужный документ от соседей по теме. Чего эти числа НЕ показывают:
насколько сложна задача «по-настоящему» — псевдозапросы легче реальных лексически и тяжелее
семантически. Что делать: запомни `BASE`. Всё, что мы построим дальше, обязано объяснить,
за счёт чего оно его обыгрывает, и обязано доказать, что обыгрывает не случайно.

<details><summary>Что именно происходит внутри Recall@k при одном релевантном документе</summary>

У нас ровно один правильный ответ на запрос, и это делает метрики проще, чем кажется, —
и одновременно капризнее.

**Recall@k вырождается в долю успехов.** Если релевантный документ один, `Recall@k` для
конкретного запроса равен либо нулю, либо единице: попал в топ-k или нет. Усреднение по запросам
даёт долю запросов, где ответ найден. Это удобно интерпретировать («в 90 % случаев ответ
в первой сотне»), но нужно понимать, что это уже не полнота в смысле «какую долю релевантного
мы собрали».

**MRR становится очень шумным.** RR принимает значения 1, 1/2, 1/3, …, 1/10 и 0 — то есть это
дискретная величина с тяжёлой массой в нуле и в единице. Её дисперсия велика по построению,
и именно поэтому доверительные интервалы в части 4 получились такими широкими. При нескольких
релевантных документах AP усредняет по попаданиям и ведёт себя спокойнее.

**Практический вывод про размер выборки.** Чем более дискретна метрика, тем больше запросов
нужно для того же разрешения. Для MRR с одним релевантным ответом полезное правило: сотня
запросов даёт различимость эффектов порядка 0,05, и это заметно хуже, чем у nDCG на плотной
разметке. Мы взяли восемьдесят и получили ровно то, что должны были: широкий интервал.

**Что можно было бы сделать иначе.** Считать не MRR, а Success@k (та же доля успехов) — она
не менее информативна при одном релевантном и чуть менее шумна, потому что не различает первое
место от третьего. Мы этого не сделали намеренно: различие между первым и третьим местом важно
для продукта, и его хотелось видеть.
</details>

In [ ]:
if METRICS_PATH.exists():
    import importlib.util
    spec = importlib.util.spec_from_file_location("m4", METRICS_PATH)
    m4 = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m4)
    probe = [0, 1, 0, 1, 0, 0, 1, 0]          # одно попадание на 2-м месте, дальше ещё два
    ours = rr_at(list(range(len(probe))), next(i for i, r in enumerate(probe) if r))
    theirs = m4.rr(probe)
    assert abs(ours - theirs) < 1e-9, (
        f"РАСХОЖДЕНИЕ реализаций: неделя 4 даёт {theirs}, сегодня {ours}. "
        "Дальше нельзя: сравнения между занятиями станут бессмысленными.")
    print(f"metrics.py недели 4 найден · RR на контрольном примере: {theirs} == {ours}")
else:
    print("metrics.py недели 4 нет -- используем сегодняшние реализации.")
    print("Это ВТОРАЯ реализация тех же метрик: сравнивая её числа с числами недели 4,")
    print("ты рискуешь измерить разницу между реализациями, а не между системами.")
RUN["metrics_from_week4"] = METRICS_PATH.exists()

**Что видно.** Либо реализация недели 4 найдена и согласуется с сегодняшней до девятого знака,
либо её нет и ноутбук говорит, чем это грозит. Сравнивать надо не два числа RR между собой —
они обязаны совпасть, — а **факт проверки с её отсутствием**: расхождение реализаций проявилось
бы не здесь, а через три недели, когда кто-то сравнил бы nDCG недели 4 с nDCG недели 9 и увидел
«эффект». Механизм тривиален и потому опасен: обе реализации выглядят правильными, обе проходят
ревью, и разница в третьем знаке принимается за результат. Чего эта проверка НЕ даёт: гарантии,
что обе реализации верны — они могут быть согласованно неверными. Что делать: держать одну
реализацию на весь курс и проверять согласие при каждом переходе между занятиями.

---

## Часть 2 · Би-энкодер: его работа — полнота, а не точность — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 2.1 | Как считается близость? | сверяем формулу с игрушкой из лекции |
| 2.2 | Обыгрывает ли он BM25 по MRR? | меряем — и получаем неожиданный ответ |
| 2.3 | Тогда зачем он нужен? | меряем Recall@k — вот где расходятся |

### Шаг 2.1 · Сверка с доской

В лекции би-энкодер разбирался на четырёхмерной игрушке: запрос «river bank», релевантный
документ про бобра у реки, нерелевантный — про банк и кредит. Проверим, что мы считаем
косинус так же.

In [ ]:
BE = json.load(open(f"{DATA_DIR}/l7-biencoder.json", encoding="utf-8"))["toy"]
q = np.array(BE["query"]["vec"], dtype=float)
dr = np.array(BE["docRel"]["vec"], dtype=float)
di = np.array(BE["docIrr"]["vec"], dtype=float)

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))) if a.any() and b.any() else 0.0

assert abs(float(q @ dr) - BE["dotRel"]) < 1e-6, "скалярное произведение разошлось с лекцией"
assert abs(cos(q, dr) - BE["cosRel"]) < 1e-4, "косинус разошёлся с лекцией"
assert abs(cos(q, di) - BE["cosIrr"]) < 1e-4, "косинус нерелевантного разошёлся"

print("оси пространства:", BE["dims"])
print(f"запрос {BE['query']['text']!r} -> {BE['query']['vec']}")
print(f"{'документ':>34} {'dot':>6} {'cos':>8}")
print(f"{BE['docRel']['text']:>34} {float(q @ dr):>6.0f} {cos(q, dr):>8.4f}")
print(f"{BE['docIrr']['text']:>34} {float(q @ di):>6.0f} {cos(q, di):>8.4f}")
print("сверка с data/l7-biencoder.json: 3 значения совпали")

**Что видно.** Косинус релевантного документа 0,8165, нерелевантного — ровно ноль. Сравнивать
надо не два косинуса по величине, а **что именно их развело**: слово `bank` есть в обоих
документах, и лексический поиск их не различит вовсе. Различает вторая координата — «география»
против «финансов», и это ровно то, ради чего существуют плотные векторы. Механизм: полисемия
разрешается контекстом, а контекст живёт в остальных измерениях вектора. Чего эта игрушка
НЕ показывает: что настоящая модель раскладывает смысл по осям с именами. У неё 384 измерения
без интерпретации, и «geography» там размазано по многим. Что делать: не искать смысл
в отдельных координатах реального эмбеддинга — искать его в расстояниях.

<details><summary>Косинус, скалярное произведение или евклид — и почему выбор не безобиден</summary>

Мы всюду пользуемся косинусом. Это не единственный вариант, и разница между вариантами
проявляется не в формуле, а в том, какие документы вылезают наверх.

**Скалярное произведение без нормировки.** Растёт с длиной вектора. У большинства энкодеров
норма вектора коррелирует с длиной текста и с его «типичностью», поэтому по dot наверх лезут
длинные и общие документы. Иногда это ровно то, что нужно: в задачах, где длинный документ
действительно полезнее, dot работает лучше косинуса. Модели MS MARCO часто обучены именно
под dot, и переключение на косинус у них портит качество.

**Косинус.** Нормирует длину, оставляя только направление. Безопасный выбор по умолчанию
и то, под что обучен `all-MiniLM-L6-v2` — в его карточке это написано прямым текстом.

**Евклидово расстояние.** На нормированных векторах монотонно связано с косинусом:
`||a−b||² = 2 − 2·cos(a,b)`. То есть на нормированных векторах евклид и косинус дают **один
и тот же порядок**, и выбирать между ними бессмысленно. На ненормированных — разный, и тогда
евклид ведёт себя как нечто среднее между dot и косинусом.

**Практическое правило.** Смотри карточку модели и используй ту метрику, под которую она
обучалась. Это одна строка документации и целый класс тихих потерь качества, если её
не прочитать. Проверить можно за минуту: посчитать MRR под обеими метриками на своих запросах
и сравнить.

**Что важно для ANN.** Индексы приближённого поиска строятся под конкретную метрику. FAISS
с `IndexFlatIP` считает скалярное произведение, `IndexFlatL2` — евклид. Подать нормированные
векторы в первый — это косинус; подать ненормированные и ждать косинуса — тихая ошибка.
Про это семинар недели 10.
</details>

<details><summary>Почему нельзя просто взять [CLS] обычного BERT — и что сделал Sentence-BERT</summary>

Естественная мысль: BERT уже даёт вектор `[CLS]`, возьмём его как эмбеддинг документа. Мысль
не работает, и история того, почему, объясняет устройство би-энкодера.

**Что не так с сырым `[CLS]`.** BERT обучен на masked language modelling и next sentence
prediction. Ни одна из этих задач не требует, чтобы близкие по смыслу предложения имели близкие
векторы `[CLS]`. Обучение вообще ничего не говорит про геометрию пространства. На практике
косинус между сырыми `[CLS]` двух предложений оказывается высоким почти всегда — векторы
собраны в узком конусе, и различать по ним нечего. Это та самая анизотропия, которой посвящена
отдельная лекция курса.

**Что сделал Sentence-BERT.** Две вещи. Первая: усреднение токенов (mean pooling) вместо
`[CLS]` — уже заметно лучше, потому что усреднение сглаживает конус. Вторая и главная:
дообучение сиамской парой на задаче, где близость **и есть** цель — контрастное обучение или регрессия
на похожесть. После такого дообучения пространство перестраивается так, что косинус начинает
означать то, что нужно.

**Почему это принципиально, а не техническая деталь.** Би-энкодер обязан кодировать документ
**до** того, как увидит запрос. Значит, вся информация о том, чему документ релевантен, должна
уместиться в один вектор, и геометрия этого пространства — единственный носитель смысла.
Кросс-энкодеру такое не нужно вовсе: он видит пару и может решать заново каждый раз. Отсюда
и разница в точности, и разница в цене.

**Что отсюда следует практически.** Никогда не бери «просто BERT» как энкодер для поиска.
Бери модель, дообученную на задаче близости, и проверяй, на каких данных она дообучалась:
`all-MiniLM-L6-v2` учился на миллиарде пар общего домена, `ms-marco-*` — на поисковых запросах,
и на твоём домене они поведут себя по-разному.
</details>

⚠️ Ловушка D · **`normalize_embeddings=True` — не косметика.** Если векторы не нормированы,
скалярное произведение это **не** косинус, и длинные документы получают больший скор просто
за норму. Мы нормируем при кодировании и дальше пользуемся скалярным произведением как
косинусом. Забудешь флаг — получишь работающий код, правдоподобные числа и тихо другое
ранжирование.

### Шаг 2.2 · Кодируем корпус и меряем

Кодирование корпуса — единственная тяжёлая операция занятия, и она делается **один раз**.
Это и есть главное экономическое свойство би-энкодера.

In [ ]:
bi = SentenceTransformer(BI_MODEL)
t0 = time.perf_counter()
EMB = bi.encode(DOCS, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
t_index = time.perf_counter() - t0

t0 = time.perf_counter()
QEMB = bi.encode(QUERIES, normalize_embeddings=True, show_progress_bar=False)
t_query = (time.perf_counter() - t0) / len(QUERIES)

SIMS = QEMB @ EMB.T
BI = [[int(d) for d in np.argsort(-SIMS[i])] for i in range(len(QUERIES))]

bi_mrr = statistics.mean(rr_at(o, g) for o, g in zip(BI, GOLD))
print(f"кодирование корпуса: {t_index:.1f} c на {len(DOCS)} документов "
      f"({t_index / len(DOCS) * 1000:.1f} мс/документ, ОДИН раз)")
print(f"кодирование запроса: {t_query * 1000:.1f} мс (каждый запрос)")
print(f"размерность вектора: {EMB.shape[1]}")
print()
print(f"{'система':>12} {'MRR@10':>9}")
print(f"{'BM25':>12} {BASE:>9.4f}")
print(f"{'би-энкодер':>12} {bi_mrr:>9.4f}   разница {bi_mrr - BASE:+.4f}")
RUN["bi_mrr10"], RUN["index_sec"], RUN["query_ms"] = bi_mrr, t_index, t_query * 1000

**Что видно.** Би-энкодер обыграл BM25 по MRR@10 на две-три сотых — то есть **почти
не обыграл**. Сравнивать надо не проценты прироста, а разницу с тем, что мы знаем про разброс:
на восьмидесяти запросах разница такого размера почти наверняка лежит внутри шума, и в части 4
мы это проверим тестом. Ожидаемая картина по механизму частично объяснена ловушкой про
псевдозапросы: они лексически щедры к BM25. Чего этот замер НЕ показывает: главного —
что би-энкодер вообще не для этого нужен. Что делать: не делать вывод «нейросеть не помогла».
Мы померяли не ту метрику, и следующий шаг показывает, какую надо было.

<details><summary>384 числа на документ: что помещается и что теряется</summary>

Размерность эмбеддинга — не свободный параметр, а решение с последствиями по всем осям сразу.

**Что она определяет.** Память индекса: миллион документов по 384 числа во float32 — это
полтора гигабайта, и это ещё до ANN-структуры. Скорость поиска: сложность точного поиска
линейна по размерности. Качество: слишком маленькая размерность физически не вмещает различия
между документами, слишком большая — обучается хуже и переобучается.

**Практические ориентиры.** 384 у MiniLM, 768 у base-моделей, 1024–1536 у больших и
у коммерческих API. Прирост качества при переходе с 384 на 768 обычно единицы процентов,
а память и время удваиваются ровно. Отсюда популярность маленьких моделей: они на удивление
конкурентоспособны.

**Матрёшечные представления.** Относительно новый приём: модель обучается так, что первые
`k` координат вектора сами по себе — осмысленный эмбеддинг меньшей размерности.
Это позволяет хранить полный вектор, а искать по обрезанному — и доуточнять только среди
кандидатов. Фактически тот же каскад, только внутри одной модели.

**Квантизация.** Второй способ уменьшить индекс: хранить не float32, а int8 или даже биты.
Потеря качества при int8 обычно в пределах процента, а память падает вчетверо. Про это
семинар недели 10, где мы будем строить ANN-индекс и мерить компромисс «память против
полноты» напрямую.

**Что стоит унести.** Размерность, точность хранения и структура индекса — три ручки одной
и той же настройки, и крутить их надо вместе, глядя на одну метрику: полноту при фиксированном
бюджете памяти и времени.
</details>

<details><summary>Экономика би-энкодера: что считается один раз, а что на каждый запрос</summary>

Разделение труда между предвычислением и временем запроса — главное, что делает поиск
возможным, и оно стоит того, чтобы посчитать его явно.

**Один раз, при индексации.** Кодирование корпуса: `N` документов × стоимость прохода модели.
У нас это секунды на полторы тысячи документов, на миллионе — часы GPU. Результат кладётся
в матрицу и живёт, пока не сменится модель. Смена модели означает **полную** переиндексацию,
и это одна из главных причин, почему модели в проде меняют редко.

**На каждый запрос.** Кодирование запроса — один проход, миллисекунды. Поиск ближайших:
у нас честное умножение матриц, то есть линейно по корпусу; в проде — ANN-индекс, о котором
целая лекция и семинар недели 10, и там сложность становится примерно логарифмической.

**Где ломается эта экономика.** Первое: обновление корпуса. Новый документ надо закодировать
и вставить в индекс; для ANN-индексов вставка бывает дорогой, а иногда требует перестройки.
Второе: персонализация. Если вектор документа должен зависеть от пользователя, предвычислить
его нельзя, и вся конструкция разваливается. Третье: длинные документы. Модель с окном
в 512 токенов не закодирует статью целиком, и приходится резать на чанки — тема недель 11 и 13,
и там появляется своя цена.

**Число, которое стоит запомнить.** Отношение стоимости индексации к стоимости запроса
у би-энкодера примерно равно размеру корпуса. Это значит, что би-энкодер окупается, когда
запросов много относительно документов. Для поиска по сотне документов и трём запросам в день
он не нужен вовсе — там дешевле кросс-энкодер в лоб.
</details>

### Шаг 2.3 · Метрика, ради которой он существует

Первая ступень каскада не обязана ставить правильный документ первым. Она обязана **не потерять
его** до второй ступени. Это Recall@k, и вот на нём картина другая.

In [ ]:
KS = (10, 25, 50, 100, 200)
rows = []
for k in KS:
    rb = statistics.mean(recall_at(o, g, k) for o, g in zip(BM, GOLD))
    rn = statistics.mean(recall_at(o, g, k) for o, g in zip(BI, GOLD))
    rows.append((k, rb, rn))

print(f"{'k':>5} {'BM25':>9} {'би-энкодер':>12} {'разница':>10}")
for k, rb, rn in rows:
    print(f"{k:>5} {rb:>9.4f} {rn:>12.4f} {rn - rb:>+10.4f}")
RUN["recall_table"] = {str(k): {"bm25": rb, "bi": rn} for k, rb, rn in rows}

**Что видно.** С ростом `k` разрыв растёт: на десятке системы почти неразличимы, на сотне
би-энкодер впереди заметно, на двух сотнях — ещё сильнее. Сравнивать надо не строки таблицы
друг с другом, а **правый столбец с нулём по мере роста k**: разница монотонно увеличивается.
Механизм в этом и состоит — BM25 упирается в лексический потолок, который мы считали на неделе 3:
документ без слов запроса он не вернёт **никогда**, сколько ни увеличивай `k`. Би-энкодер
такого потолка не имеет и продолжает подбирать нужное на глубине. Чего эта таблица НЕ показывает:
что би-энкодер лучше «вообще» — по MRR@10 он был почти равен. Что делать: запомнить разделение
труда. Первая ступень отвечает за то, чтобы ответ был среди кандидатов; кто поставит его первым,
решается на следующей ступени.

⚠️ Ловушка B · **Мерить первую ступень по MRR — значит мерить не то.** Если бы мы остановились
на предыдущем шаге, вывод был бы «би-энкодер не нужен, BM25 не хуже». Он неверен, и неверен
именно из-за выбора метрики. Каждая ступень каскада меряется своей метрикой: первая — полнотой,
последняя — точностью в верхушке.

---

## Часть 3 · Кросс-энкодер судит — 30 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 3.1 | Как он устроен? | сверяем игрушку с лекцией |
| 3.2 | Что даёт переранжирование? | меряем на глубине 25 |
| 3.3 | Сколько он может дать в принципе? | **замер без модели**: потолок переранжирования |

In [ ]:
CE = json.load(open(f"{DATA_DIR}/l7-crossencoder.json", encoding="utf-8"))["toy"]

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

w, b = np.array(CE["w"], dtype=float), CE["b"]
logit_rel = float(w @ np.array(CE["clsRel"], dtype=float) + b)
logit_neg = float(w @ np.array(CE["clsNeg"], dtype=float) + b)

assert abs(logit_rel - CE["logitRel"]) < 1e-6, "логит релевантного разошёлся с лекцией"
assert abs(sigmoid(logit_rel) - CE["scoreRel"]) < 1e-4, "сигмоида разошлась с лекцией"
assert abs(logit_neg - CE["logitNeg"]) < 1e-6, "логит нерелевантного разошёлся"

print("запрос:", CE["qTokens"], "· документ:", CE["dTokens"])
print(f"{'пара':>14} {'логит':>8} {'после сигмоиды':>16}")
print(f"{'релевантная':>14} {logit_rel:>8.2f} {sigmoid(logit_rel):>16.4f}")
print(f"{'нерелевантная':>14} {logit_neg:>8.2f} {sigmoid(logit_neg):>16.4f}")
print("сверка с data/l7-crossencoder.json: 3 значения совпали")

**Что видно.** Голова кросс-энкодера — это одна линейная свёртка вектора `[CLS]` плюс сигмоида,
и вся её сложность в том, что попадает в `[CLS]`. Сравнивать надо не логиты с вероятностями,
а **логиты между собой**: порядок задаётся именно ими, а сигмоида монотонна и порядка
не меняет вообще. Механизм важен для практики: если тебе нужен только ранжирующий порядок,
сигмоиду можно не считать, а если нужен порог отсечения — считать обязательно, потому что
порог по логиту и порог по вероятности это разные вещи. Чего эта игрушка НЕ показывает: работы
внимания между токенами запроса и документа — именно она даёт кросс-энкодеру точность,
и именно она делает его непредвычислимым. Что делать: помнить, что `predict` у настоящей
модели вернёт **логиты**, а не вероятности.

<details><summary>Что даёт кросс-энкодеру точность — и почему это нельзя предвычислить</summary>

Разница между би- и кросс-энкодером не в размере модели: у нас они обе на шесть слоёв
и двадцать два миллиона параметров. Разница в том, что видит внимание.

**Би-энкодер.** Документ кодируется в отсутствие запроса. Модель обязана заранее решить, какие
аспекты документа сохранить в 384 числах, не зная, о чём спросят. Документ про историю
и химию бензина сожмётся в один вектор, и на запросе про химию он будет наполовину «про
историю». Это фундаментальное ограничение: одно представление на все возможные запросы.

**Кросс-энкодер.** Токены запроса и документа попадают в один вход, и на каждом слое внимание
считает, какие слова документа релевантны каким словам запроса. Слово `flood` в запросе может
подсветить `flood` в документе и заодно понять из соседей, что речь про реку, а не про поток
данных. Никакого сжатия в фиксированный вектор нет — есть прямое сопоставление.

**Почему предвычислить нельзя, строго.** Представление документа в кросс-энкодере зависит
от запроса начиная с первого слоя внимания. Кэшировать можно только то, что от запроса
не зависит, а это ничего. Существуют промежуточные архитектуры — ColBERT кэширует
потокенные векторы документа и откладывает взаимодействие на самый конец, — и о них лекция L12
и семинар недели 9. Они и есть попытка забрать часть точности кросс-энкодера, сохранив
предвычислимость.

**Что это значит для нашего результата.** Кросс-энкодер должен был бы дать заметный выигрыш,
и на MS MARCO он его даёт (+22,8 % относительных в цитируемых числах). У нас интервал накрывает
ноль. Наиболее вероятная причина не в модели, а в задаче: псевдозапросы разрешаются лексически,
а тонкое различение, ради которого нужен кросс-энкодер, там почти не требуется.
</details>

⚠️ Ловушка D · **`CrossEncoder.predict` возвращает логиты.** У `ms-marco-MiniLM-L-6-v2` они
лежат примерно в диапазоне от −11 до +11, и они — не вероятности. Для ранжирования это
безразлично, для порога «показывать или нет» — нет. Сравнивать логиты между **разными** моделями
нельзя вовсе: у каждой своя шкала.

### Шаг 3.2 · Переранжирование на глубине 25

In [ ]:
ce = CrossEncoder(CE_MODEL)

def rerank(depth, batch=32):
    # -> (порядки, скоры в том же порядке, мс/запрос). Скоры возвращаем ВМЕСТЕ с порядком:
    # порядок отвечает на "кто выше", скоры -- на "насколько", второе нужно для слияния.
    out, sc_out, t0 = [], [], time.perf_counter()
    for i, qtext in enumerate(QUERIES):
        cand = BI[i][:depth]
        scores = ce.predict([(qtext, DOCS[d]) for d in cand],
                            batch_size=batch, show_progress_bar=False)
        order = np.argsort(-scores)
        out.append([int(cand[j]) for j in order])
        sc_out.append([float(scores[j]) for j in order])
    return out, sc_out, (time.perf_counter() - t0) / len(QUERIES)

RERANK25, SCORES25, t_re25 = rerank(25)
mrr25 = statistics.mean(rr_at(o, g) for o, g in zip(RERANK25, GOLD))

print(f"{'система':>22} {'MRR@10':>9} {'мс/запрос':>11}")
print(f"{'BM25':>22} {BASE:>9.4f} {RUN['bm25_ms']:>11.1f}")
print(f"{'би-энкодер':>22} {bi_mrr:>9.4f} {RUN['query_ms']:>11.1f}")
print(f"{'каскад, глубина 25':>22} {mrr25:>9.4f} {t_re25 * 1000:>11.1f}")
print(f"\nприрост каскада к BM25: {(mrr25 / BASE - 1) * 100:+.1f}% относительных")
RUN["rerank25_mrr10"], RUN["rerank25_ms"] = mrr25, t_re25 * 1000

**Что видно.** Каскад даёт около семнадцати процентов относительного прироста к BM25 и стоит
почти на два порядка дороже по времени на запрос: сотня с лишним миллисекунд у каскада против 1,9 мс у одного BM25. Сравнивать надо не проценты сами по себе, а
**прирост с ценой и с разбросом**: проценты выглядят убедительно ровно до того момента, пока
не посчитан доверительный интервал, и это мы сделаем в части 4. Ожидаемая картина по механизму:
кросс-энкодер видит взаимодействие слов запроса и документа напрямую и потому лучше различает
близких кандидатов. Чего этот замер НЕ показывает: устойчивости прироста — восемьдесят запросов
это мало, и в прошлый раз мы уже видели, как «значимо» превращается в интервал, упирающийся
в ноль. Что делать: не называть это число вслух до части 4.

### Шаг 3.3 · Замер без модели: потолок переранжирования

**Замер без модели.** Сколько кросс-энкодер может дать **в принципе**? Ровно столько, сколько
позволяет первая ступень: если нужного документа нет среди кандидатов, никакое переупорядочивание
его не создаст. Потолок MRR@10 при глубине `d` равен доле запросов, где ответ попал в топ-`d`
би-энкодера, — то есть `Recall@d`. Это свойство первой ступени, и от кросс-энкодера оно
не зависит вовсе.

In [ ]:
print(f"{'глубина':>9} {'потолок (Recall@d)':>20}")
ceilings = {}
for d in DEPTHS:
    ceilings[d] = statistics.mean(recall_at(o, g, d) for o, g in zip(BI, GOLD))
    print(f"{d:>9} {ceilings[d]:>20.4f}")
print(f"\nдостигнуто на глубине 25: {mrr25:.4f} из потолка {ceilings[25]:.4f} "
      f"-- реализовано {mrr25 / ceilings[25]:.0%}")
RUN["ceilings"] = {str(k): v for k, v in ceilings.items()}

**Что видно.** Потолок растёт с глубиной, а реализовано от него около двух третей. Сравнивать
надо не достигнутое с единицей, а **достигнутое с потолком своей глубины**: это разделяет две
совершенно разные причины неудачи. Если потолок низок — виновата первая ступень, и чинить надо
её, увеличивая глубину или меняя ретривер. Если потолок высок, а реализовано мало — виноват
кросс-энкодер, и чинить надо его, дообучая. Механизм прямой: потолок это `Recall@d`, то есть
чистая характеристика отбора. Чего этот замер НЕ показывает: что будет, если поднять потолок —
рост потолка не означает роста качества, и именно это мы проверим в следующей части. Что делать:
считать этот потолок **всегда**, когда строишь каскад. Он стоит одну строку и сразу говорит,
какую ступень чинить.

<details><summary>Потолок каскада: полная формула и что делать с каждым из трёх случаев</summary>

Потолок, который мы посчитали одной строкой, — частный случай общего утверждения, полезного
при любом числе ступеней.

**Общая форма.** Для каскада из ступеней с глубинами `d1 > d2 > ... > dk` потолок итоговой
метрики задаётся полнотой **первой** ступени на её глубине: `Recall@d1`. Каждая следующая
ступень может только переупорядочивать то, что получила, и потому не способна поднять полноту.
Если ступеней три, узкое место всё равно первое: вторая и третья ограничены тем же множеством.

**Три случая и что делать в каждом.**

*Потолок низок, реализовано много.* Судья выжимает почти всё, что ему дают. Единственный путь —
улучшать отбор: увеличивать глубину, менять ретривер, добавлять вторую первую ступень
(лексическую рядом с плотной — это ровно гибрид из недели 9).

*Потолок высок, реализовано мало.* Кандидаты есть, судья не различает. Путь — дообучать судью
на своём домене, менять модель, добавлять признаки. Увеличение глубины здесь не поможет
и, как мы видели, повредит.

*Потолок высок, реализовано много.* Каскад работает; дальнейший рост требует либо более
точной модели, либо признания, что задача решена в пределах разметки.

**Наш случай.** Потолок на рабочей глубине заметно ниже единицы, а доля реализованного падает
с глубиной. Формально это второй случай, но с оговоркой: падение доли означает не только
слабость судьи, но и то, что добавляемые кандидаты систематически хуже. Оба лечения — дообучение
судьи и улучшение отбора — работают, и выбирать между ними надо по стоимости, а не по этой
таблице.

**Что стоит унести.** Две строки кода — `Recall@d` и `MRR/Recall@d` — заменяют часы гаданий
о том, где узкое место. Считай их первыми, до любых экспериментов с моделями.
</details>

⚠️ Ловушка C · **Потолок — не обещание.** Recall@100 равен 0,9, но это не значит, что при
глубине 100 MRR@10 приблизится к 0,9. Потолок ограничивает сверху и ничего не гарантирует;
между ним и результатом лежит вся способность кросс-энкодера отличать нужное от похожего.

---

## Часть 4 · Глубина: качество против задержки — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 4.1 | Растёт ли качество с глубиной? | прогон по четырём глубинам |
| 4.2 | Сколько это стоит? | те же глубины в миллисекундах |
| 4.3 | Отличимы ли различия от шума? | парный тест по методике недели 4 |

Здесь принимается единственное настоящее инженерное решение занятия.

In [ ]:
sweep = {}
for d in DEPTHS:
    orders, scores, ms = rerank(d)
    per_query = [rr_at(o, g) for o, g in zip(orders, GOLD)]
    sweep[d] = {"mrr": statistics.mean(per_query), "ms": ms * 1000,
                "ceiling": ceilings[d], "per_query": per_query, "scores": scores}

print(f"{'глубина':>9} {'MRR@10':>9} {'потолок':>9} {'реализ.':>9} {'мс/запрос':>11}")
for d in DEPTHS:
    s = sweep[d]
    print(f"{d:>9} {s['mrr']:>9.4f} {s['ceiling']:>9.4f} "
          f"{s['mrr'] / s['ceiling']:>9.0%} {s['ms']:>11.1f}")
RUN["sweep"] = {str(d): {k: v for k, v in s.items() if k not in ("per_query", "scores")}
                for d, s in sweep.items()}

**Что видно.** Потолок исправно растёт с глубиной, а MRR@10 — **нет**: он выходит на полку
уже на глубине 10–25 и дальше стоит на месте, иногда чуть проседая. Сравнивать надо не строки
таблицы друг с другом, а **столбец потолка со столбцом качества**: они расходятся, и это главный
факт занятия. Механизм понятен, если подумать, кого мы добавляем при углублении: с ростом `d`
в набор кандидатов попадают документы, которые би-энкодер оценил хуже, — то есть в среднем
менее подходящие. Правильный ответ среди них появляется редко, а вот шанс, что кросс-энкодер
поднимет один из них выше правильного, растёт с каждым новым кандидатом. Чего эта таблица
НЕ показывает: различимы ли эти колебания — следующие две ячейки. Что делать: не увеличивать
глубину «на всякий случай». Она платная и, судя по этой таблице, бесполезная за некоторым
порогом.

<details><summary>Почему углубление вредит: механизм и что с этим делают в проде</summary>

Наблюдение «глубже — не лучше» кажется парадоксальным: мы же добавляем кандидатов, среди
которых может оказаться правильный. Разберём по шагам, почему это не помогает.

**Что происходит при переходе с глубины 25 на 50.** Добавляются двадцать пять документов,
которые би-энкодер поставил на места с 26 по 50. Вероятность, что правильный ответ там,
равна разнице `Recall@50 − Recall@25` — у нас это около шести процентов запросов. То есть
в 94 % случаев мы добавили только конкурентов.

**Что делает с ними судья.** Кросс-энкодер оценивает каждого независимо и не знает, что новые
кандидаты «хуже по мнению первой ступени». Если хотя бы один из двадцати пяти получит скор выше
правильного ответа, метрика упадёт. При достаточно большом наборе кандидатов такое событие
почти неизбежно — это тот же эффект, что множественные сравнения из недели 4, только вместо
гипотез перебираются документы.

**Баланс.** Выигрыш линеен по приросту полноты, проигрыш растёт с числом добавленных
конкурентов. На малых глубинах полнота растёт быстро и выигрывает; дальше кривая полноты
выполаживается, а конкурентов становится всё больше, и знак меняется.

**Что делают в проде.** Первое: не углубляют, а улучшают отбор — гибрид лексического
и плотного даёт больший прирост полноты на той же глубине. Второе: используют скор первой
ступени как признак второй, чтобы судья знал, что кандидат пришёл с 90-го места, — это простая
и часто недооценённая мера. Третье: калибруют судью так, чтобы он не поднимал документы,
в которых не уверен, — то есть добавляют порог, а не только порядок.

**Проверка, которую стоит сделать самому.** Возьми глубину 100 и посмотри, на каких запросах
качество упало относительно глубины 25. Почти наверняка это будут запросы, где правильный ответ
и так был найден, а испортил его конкретный документ, поднятый судьёй. Такой разбор занимает
десять минут и говорит больше, чем весь прогон.
</details>

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 3.4))
ax1.plot(DEPTHS, [sweep[d]["ceiling"] for d in DEPTHS], "o--", color="#2E7D52",
         label="потолок (Recall@d)")
ax1.plot(DEPTHS, [sweep[d]["mrr"] for d in DEPTHS], "o-", color="#3B6FD4", label="MRR@10 каскада")
ax1.axhline(BASE, color="#B4521F", ls=":", label=f"BM25 {BASE:.3f}")
ax1.set_xlabel("глубина переранжирования"); ax1.set_ylabel("качество"); ax1.set_ylim(0, 1)
ax2 = ax1.twinx()
ax2.bar(DEPTHS, [sweep[d]["ms"] for d in DEPTHS], width=6, alpha=.18, color="#111")
ax2.set_ylabel("мс/запрос (столбики)")
ax1.legend(loc="lower right", fontsize=9)
plt.title("Потолок растёт, качество стоит, цена растёт линейно")
plt.tight_layout(); plt.show()

**Что видно.** Сравнивать надо не три кривые попарно, а **зелёную с синей**: зелёная (потолок)
уверенно поднимается, синяя (реальное качество) идёт горизонтально. Серые столбики растут
линейно по глубине — цена честно пропорциональна числу пар. Ожидаемая картина ровно такая
по механизму из предыдущего разбора: расширение набора кандидатов добавляет шума быстрее,
чем сигнала. Чего график НЕ показывает: доверительных интервалов у синей кривой — визуально
её колебания кажутся содержательными, а они, скорее всего, шум. Что делать: смотреть на этот
график как на **три** утверждения, а не одно, и проверять каждое отдельно. Проверим главное.

### Шаг 4.3 · Шум раньше эффекта

Методика недели 4, применённая к собственным числам: разница на каждом запросе, среднее,
разброс, доверительный интервал.

In [ ]:
def paired(a, b, label):
    d = [x - y for x, y in zip(a, b)]
    m = statistics.mean(d)
    se = statistics.stdev(d) / math.sqrt(len(d))
    lo, hi = m - 2.0 * se, m + 2.0 * se
    verdict = "различимо" if lo > 0 or hi < 0 else "НЕ отличимо от нуля"
    print(f"{label:34} Δ={m:+.4f}  ДИ95≈[{lo:+.4f}; {hi:+.4f}]  "
          f"{sum(x > 0 for x in d):>2}/{sum(x < 0 for x in d):<2}  {verdict}")
    return {"delta": m, "ci": [lo, hi], "distinguishable": lo > 0 or hi < 0}

bm_pq = [rr_at(o, g) for o, g in zip(BM, GOLD)]
bi_pq = [rr_at(o, g) for o, g in zip(BI, GOLD)]
print(f"{'сравнение':34} {'разница':>9}  {'интервал':>22}  {'+/-':>5}  вердикт")
RUN["paired"] = {
    "bi_vs_bm25": paired(bi_pq, bm_pq, "би-энкодер против BM25"),
    "cascade25_vs_bm25": paired(sweep[25]["per_query"], bm_pq, "каскад@25 против BM25"),
    "cascade25_vs_bi": paired(sweep[25]["per_query"], bi_pq, "каскад@25 против би-энкодера"),
    "d100_vs_d25": paired(sweep[100]["per_query"], sweep[25]["per_query"],
                          "глубина 100 против глубины 25"),
}

**Что видно.** Ни одно сравнение не отличимо от нуля: все интервалы накрывают ноль. Сравнивать
надо не средние приросты, а **границы интервалов**: каскад «даёт +17 %» к BM25, и при этом данные
совместимы с тем, что он не даёт ничего. Ожидаемая картина по механизму — восемьдесят запросов
при таком разбросе просто не дают разрешения: на неделе 4 мы считали, что эффект такого размера
требует порядка пятидесяти-ста запросов **при вдвое меньшем разбросе**, а MRR по одному
релевантному документу скачет от 0 до 1 и разброс даёт огромный. Чего этот вывод НЕ показывает:
что каскад бесполезен — отсутствие доказательства не есть доказательство отсутствия. Что делать:
различать два разных «не знаем». Про качество мы не знаем. Про **глубину** мы знаем: разница
между 25 и 100 равна нулю с узким интервалом, а цена отличается втрое. Это решение принимается
уверенно, и оно — брать наименьшую глубину, на которой качество уже вышло на полку. Какую
именно, найдёт задание 1, и ответ может оказаться меньше двадцати пяти.

<details><summary>Что делать, когда интервал накрывает ноль — четыре честных выхода</summary>

Мы получили результат «не отличимо от нуля» на всех сравнениях качества. Это не тупик, это
развилка, и на ней есть ровно четыре честных пути.

**Первый: увеличить выборку.** Разрешение растёт как корень из числа запросов. Чтобы сузить
интервал вдвое, нужно вчетверо больше запросов — у нас это триста двадцать вместо восьмидесяти.
Псевдозапросы генерируются бесплатно, так что это самый дешёвый путь, и он же самый честный
при условии, что новые запросы берутся из той же популяции, а не подбираются.

**Второй: уменьшить разброс.** Метрика с меньшей дисперсией даёт то же разрешение на меньшей
выборке. Вместо MRR по одному релевантному ответу — nDCG по градуированной разметке, или хотя бы
несколько релевантных документов на запрос. Плата: нужна разметка.

**Третий: признать, что эффект мал, и решать по стоимости.** Если данные говорят, что выигрыш
лежит между нулём и семью сотыми, а цена известна точно и велика, решение принимается по цене.
Это нормальный инженерный вывод, и он часто правильный: не всё, что не доказано, надо внедрять.

**Четвёртый: сменить вопрос.** Мы не смогли различить качество, но уверенно различили
бесполезность глубины. Часто именно так и бывает: исходный вопрос не разрешается имеющимися
данными, а соседний — разрешается, и ответ на него полезен.

**Чего делать нельзя.** Нельзя добирать запросы, пока интервал не перестанет накрывать ноль,
и останавливаться в этот момент. Это называется optional stopping, и оно гарантированно даёт
ложноположительный результат при достаточном терпении. Размер выборки фиксируется до замера.
</details>

⚠️ Ловушка C · **«+17 % относительных» — самая честная ложь на этом занятии.** Число посчитано
верно. Интервал накрывает ноль. Оба утверждения истинны одновременно, и первое без второго
вводит в заблуждение. Так выглядит большинство отчётов об улучшении поиска.

⚠️ Ловушка E · **Задержка померяна без прогрева и на CPU.** Первый вызов модели включает
её инициализацию, батч 32 при глубине 10 недозагружен, а на GPU соотношения будут другими.
Переносить наши миллисекунды нельзя; переносится **линейность** цены по глубине.

⚠️ Ловушка F · **Опубликованные числа этой пары моделей — не наши.** В `data/l7-msmarco.json`
лежит MRR@10 = 0,5482 у ретривера и 0,6732 после переранжирования на выборке MS MARCO.
Это другая коллекция, другие запросы и другая разметка. Сравнивать наши 0,4987 с их 0,6732
нельзя ни в какую сторону.

In [ ]:
MS = json.load(open(f"{DATA_DIR}/l7-msmarco.json", encoding="utf-8"))
print("ВНЕШНИЕ числа (цитируются, НЕ наш замер) -- MS MARCO passage, выборка из лекции:")
print(f"   корпус {MS['subset']['corpusSize']} пассажей, {MS['subset']['nQueries']} запросов")
print(f"   ретривер {MS['retrieve']['model']}: MRR@10 = {MS['retrieve']['mrrAt10']}")
print(f"   + кросс-энкодер, глубина {MS['rerank']['rerankDepth']}: "
      f"MRR@10 = {MS['rerank']['mrrAt10']}, nDCG@10 = {MS['rerank']['ndcgAt10']}")
print(f"   прирост от переранжирования: "
      f"{(MS['rerank']['mrrAt10'] / MS['retrieve']['mrrAt10'] - 1) * 100:+.1f}% относительных")
print()
print("НАШИ числа (20NG, псевдозапросы, та же пара моделей):")
print(f"   би-энкодер: MRR@10 = {bi_mrr:.4f}")
print(f"   + кросс-энкодер, глубина 25: MRR@10 = {sweep[25]['mrr']:.4f}")
print(f"   прирост: {(sweep[25]['mrr'] / bi_mrr - 1) * 100:+.1f}% относительных, "
      f"интервал накрывает ноль")
RUN["cited_msmarco"] = {"retrieve": MS["retrieve"]["mrrAt10"], "rerank": MS["rerank"]["mrrAt10"]}

**Что видно.** Относительный прирост от переранжирования на MS MARCO и у нас — величины одного
порядка, притом что абсолютные значения различаются сильно. Сравнивать надо именно **отношения**,
а не абсолютные MRR: у них другая коллекция, другая разметка и запросы, написанные людьми,
а не вырезанные из документов. Механизм совпадения отношений содержательный: переранжирование
даёт похожий относительный выигрыш там, где первая ступень уже нашла кандидатов. Чего это
сопоставление НЕ доказывает: что наш замер «подтверждён» внешним. Два числа, посчитанные
на разных данных, не подтверждают друг друга — они лишь не противоречат. Что делать: держать
внешние и свои числа в **разных** таблицах и никогда не смешивать в одной строке. Мы их
и печатаем двумя блоками с явными заголовками.

<details><summary>Почему модель, отличная на MS MARCO, проигрывает BM25 на чужой коллекции</summary>

Наши числа выглядят разочаровывающе рядом с цитируемыми. Это не наша неудача и не случайность —
это самый воспроизводимый результат в современном поиске, и у него есть имя: провал переноса.

**Что показал BEIR.** Работа Thakur et al. собрала восемнадцать разнородных коллекций и прогнала
по ним модели, обученные на MS MARCO. Результат: на большинстве коллекций плотные ретриверы
уступают BM25, иногда заметно. Модель, выигрывающая на домашней коллекции с большим отрывом,
на чужой оказывается хуже формулы семидесятых годов.

**Почему так происходит.** Плотный ретривер учит соответствие между **распределением запросов**
и **распределением документов** своей обучающей выборки. MS MARCO — это короткие вопросы
от пользователей Bing к веб-пассажам. Перенеси на научные абзацы, юридические тексты
или наши почтовые треды — и распределения не совпадают ни по длине, ни по стилю, ни по лексике.
BM25 при этом ничего не учил и потому ничего не теряет: он одинаково посредствен везде.

**Почему это фундаментально, а не вопрос данных.** Би-энкодер обязан решить заранее, какие
аспекты документа сохранить в вектор. Что сохранять — он выучил из обучающих запросов. Придут
запросы другого типа — сохранено окажется не то. У BM25 такого решения нет вовсе: он хранит
все слова.

**Что делают на практике.** Первое и главное: гибрид. Лексический и плотный скор складывают,
и результат почти всегда не хуже лучшего из двух — потому что промахи у них разного типа.
Это неделя 9. Второе: дообучение на своём домене, хотя бы на псевдозапросах вроде наших.
Третье: не выбрасывать BM25 никогда, даже если плотный выигрывает, — он дёшев и служит
страховкой на запросах с редкими терминами.

**Что это значит для нашего результата.** Наш би-энкодер не обыграл BM25 по MRR@10, и это
ожидаемо: 20NG не похож на MS MARCO ничем. Правильный вывод не «модель плохая», а «модель
не переносится», и лечится он ровно теми тремя способами выше.
</details>

---

## Задания — 15 мин

**Про самопроверку честно:** пройденная самопроверка не гарантирует, что задание сделано
осмысленно, но проваленная гарантирует, что где-то ошибка.

### Задание 1 · Найди рабочую глубину

**Что сделать.** Определи наименьшую глубину из `DEPTHS`, после которой качество перестаёт
расти, и посчитай, во что обходится каждая следующая. Формально: найди `best_depth` —
минимальную глубину, чьё MRR@10 отличается от максимального по прогону меньше чем на 0,01,
и `overpay` — во сколько раз дороже самая большая глубина по сравнению с `best_depth`.

**Что нужно получить.** `best_depth` (`int` из `DEPTHS`), `overpay` (`float`).

**Подсказка.** `max(s["mrr"] for s in sweep.values())`, дальше первый подходящий из `DEPTHS`.

**Прочитай до запуска.** Исходы:
* `best_depth` = 10 или 25 — качество выходит на полку рано, и это типичный результат;
* `best_depth` = 100 — качество растёт до конца прогона, значит глубину стоит расширить дальше;
* `overpay` близко к единице — цена почти не зависит от глубины, проверь, не в SMOKE ли ты.

**Формулировка вывода.** Не «оптимальная глубина равна N», а: **при каком условии** имеет смысл
платить за большую глубину, и как это условие проверить одним замером.

In [ ]:
# --- твой код: ЗАДАНИЕ 1 ---
best_depth = ...
overpay = ...
# --- конец ---

assert best_depth in DEPTHS, "best_depth должен быть одной из глубин свипа"
assert overpay >= 1.0, "overpay -- это ВО СКОЛЬКО РАЗ дороже, значит не меньше единицы"
best_mrr, max_mrr = sweep[best_depth]["mrr"], max(s["mrr"] for s in sweep.values())
assert max_mrr - best_mrr < 0.01 + 1e-9, \
    f"на глубине {best_depth} качество ниже максимума на {max_mrr - best_mrr:.4f} -- это не полка"
assert all(max_mrr - sweep[d]["mrr"] >= 0.01 - 1e-9 for d in DEPTHS if d < best_depth), \
    "нашлась глубина МЕНЬШЕ, которая тоже на полке -- ты выбрал не минимальную"
print(f"рабочая глубина {best_depth}: MRR@10 {best_mrr:.4f} при {sweep[best_depth]['ms']:.0f} мс")
print(f"максимальная глубина {max(DEPTHS)} дороже в {overpay:.1f} раза, качество "
      f"{sweep[max(DEPTHS)]['mrr'] - best_mrr:+.4f}")
RUN["task1"] = {"best_depth": best_depth, "overpay": overpay}

### Задание 2 · Кого чинить: разведчика или судью

**Тезис.** *Разрыв между потолком и достигнутым говорит, какую ступень чинить.* Посчитай для
каждой глубины долю реализованного потолка и найди, где она максимальна.

**Что сделать.** Собери `realized` — словарь `{глубина: MRR/потолок}` — и определи `fix_stage`.
Правило: если доля реализованного **падает** с глубиной, значит судья не справляется с потоком,
и чинить надо `"reranker"`; если держится или растёт, а потолок при этом низок — узкое место
в отборе, и чинить надо `"retriever"`.

**Что нужно получить.** `realized` (`dict`), `fix_stage` (`str`).

**Подсказка.** Потолки уже посчитаны в `ceilings`.

**Прочитай до запуска.** Все исходы содержательны:
* доля реализованного падает с глубиной — судья тонет в кандидатах, углублять вредно;
* доля растёт — судья справляется, и потолок стоит поднимать;
* доля постоянна — обе ступени масштабируются согласованно, редкий и приятный случай.

**Формулировка вывода.** Не «надо чинить X», а: **какое наблюдение** заставило бы тебя поменять
решение на противоположное.

<details><summary>Шесть типов ловушек этого занятия — и чем оно отличается от двух предыдущих</summary>

Собери сегодняшние ловушки вместе, и видно смещение центра тяжести относительно недель 3 и 4.

**A · данных.** Псевдозапросы делятся словарём с целевым документом — смещение в пользу BM25.
Один релевантный документ на запрос — упрощение, меняющее поведение метрик полноты.

**B · метрики.** Мерить первую ступень по MRR — значит мерить ту часть её работы, которую
вторая ступень выбросит. Если бы мы остановились на шаге 2.2, вывод был бы прямо
противоположным правильному.

**C · интерпретации.** «+17 % относительных» при интервале, накрывающем ноль. Потолок как
обещание, хотя он только ограничение сверху.

**D · инструмента.** `normalize_embeddings=True`, без которого скалярное произведение перестаёт
быть косинусом. `CrossEncoder.predict`, возвращающий логиты, а не вероятности.

**E · замера.** Задержка без прогрева, на CPU, с недозагруженным батчем на малых глубинах.

**F · переноса.** Цитируемые 12 мс на пару, из которых нельзя умножением получить стоимость
глубины. Числа MS MARCO, которые нельзя сравнивать с нашими ни в какую сторону.

**Чем это занятие отличается.** На неделе 3 ловушки были в основном про инструмент: библиотека
считает не то. На неделе 4 — про интерпретацию: число верное, вывод нет. Сегодня добавился
новый тип, которого раньше не было: **ловушки уровня системы**. Выбор метрики для ступени,
глубина отбора, что предвычислимо, а что нет — это решения, где ошибка не видна ни в одной
отдельной ячейке. Она проявляется только тогда, когда смотришь на конструкцию целиком.

Отсюда правило занятия: **сначала нарисуй, из чего состоит система и что чем ограничено,
и только потом считай числа.** Потолок переранжирования — одна строка кода, но чтобы её
написать, надо было понять, что первая ступень ограничивает всё остальное. Никакой замер
этого не подскажет.
</details>

In [ ]:
# --- твой код: ЗАДАНИЕ 2 ---
realized = ...
fix_stage = ...
# --- конец ---

assert set(realized) == set(DEPTHS), "нужна доля для КАЖДОЙ глубины свипа"
assert all(0.0 <= v <= 1.0 for v in realized.values()), \
    "доля реализованного лежит в [0,1] -- иначе перепутаны местами MRR и потолок"
assert fix_stage in ("retriever", "reranker"), "укажи ступень строкой"
falling = realized[max(DEPTHS)] < realized[min(DEPTHS)]
assert (fix_stage == "reranker") == falling, \
    "вердикт не согласуется с трендом доли реализованного -- перечитай правило"
print(f"{'глубина':>9} {'реализовано':>13}")
for d in DEPTHS:
    print(f"{d:>9} {realized[d]:>13.1%}")
print(f"чинить: {fix_stage} · доля реализованного "
      f"{realized[min(DEPTHS)]:.0%} -> {realized[max(DEPTHS)]:.0%} с ростом глубины")
RUN["task2"] = {"realized": {str(k): v for k, v in realized.items()}, "fix_stage": fix_stage}

### Задание 3 · Словами: почему первая ступень меряется полнотой

**Что сделать.** В части 2 би-энкодер почти не обыграл BM25 по MRR@10, но заметно обошёл
по Recall@100. Ответь **словами** на два вопроса:

1. Почему для первой ступени каскада Recall@k — правильная метрика, а MRR — нет? Свяжи ответ
   с потолком из части 3.
2. Приведи ситуацию, в которой первую ступень всё-таки надо мерить точностью, а не полнотой,
   и объясни, чем эта ситуация отличается.

**Прочитай до запуска.** `assert` проверяет объём и что заглушка заменена. Ответ без слова
«потолок» и без конкретного примера во втором пункте проверку пройдёт и ревью не пройдёт.

**Формулировка вывода.** Не «recall важнее», а: при каком устройстве системы утверждение
переворачивается.

In [ ]:
# --- твой код: ЗАДАНИЕ 3 ---
ANSWER = """
Впиши ответ сюда: минимум 70 слов, оба пункта, во втором -- конкретный пример.
"""
# --- конец ---

assert len(ANSWER.split()) >= 70, "ответ короче 70 слов -- два пункта так не уместить"
assert "Впиши ответ" not in ANSWER, "заглушка не заменена"
assert "потолок" in ANSWER.lower(), "первый пункт просит связать ответ с потолком"
print(f"ответ принят: {len(ANSWER.split())} слов")

---

## Итог занятия — 5 мин

* Посчитали, почему одноступенчатый кросс-энкодер невозможен: разница в порядках, а не в разах.
* Увидели, что би-энкодер почти не обыгрывает BM25 по MRR — и заметно обыгрывает по Recall@k.
  Первая ступень отвечает за полноту, и мерить её надо полнотой.
* Померяли **потолок переранжирования** — одну строку кода, которая сразу говорит, какую
  ступень чинить.
* Прогнали ряд глубин и обнаружили, что потолок растёт, а качество стоит: углубление
  добавляет шума быстрее, чем сигнала.
* Применили методику недели 4 к собственным числам и получили честный результат: прирост
  каскада **не отличим от нуля** на восьмидесяти запросах, а вот бесполезность глубины 100
  против 25 — отличима, и это решение можно принять.

**Ограничение нашего замера, которое надо назвать вслух.** Псевдозапросы лексически щедры
к BM25 и не похожи на пользовательские. Один релевантный документ на запрос делает MRR
прыгающей величиной с огромным разбросом. Восемьдесят запросов — мало. Всё это смещает выводы
в сторону «разницы не видно», и правильная формулировка результата — «наш замер не обладает
разрешением», а не «каскад не работает».

**Что мы будем и чего не будем замерять дальше.** На неделе 9 к этим двум ступеням добавится
третья, и вопрос станет прежним: отличим ли выигрыш от шума. Ответ будет зависеть от того,
наберём ли мы к тому времени больше запросов.

In [ ]:
np.save(ARTIFACTS / "corpus_emb.npy", EMB)
(ARTIFACTS / "cascade.json").write_text(json.dumps({
    "queries": QUERIES, "gold": GOLD,
    # Порядок И скоры обеих ступеней. Порядок отвечает «кто выше», скоры -- «насколько»,
    # и без вторых нельзя построить слияние, не пересчитав ступени заново.
    "bm25_top100": [o[:100] for o in BM],
    "bm25_scores100": [s[:100] for s in BM_SCORES],
    "bi_top200": [o[:200] for o in BI],
    "bi_scores200": [[float(SIMS[i][d]) for d in BI[i][:200]] for i in range(len(QUERIES))],
    "rerank25": RERANK25,
    "rerank25_scores": SCORES25,
    "run": RUN,
}, ensure_ascii=False), encoding="utf-8")

RUN["finished"] = True
(ARTIFACTS / "run-cascade.json").write_text(
    json.dumps(RUN, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"эмбеддинги -> {ARTIFACTS / 'corpus_emb.npy'} ({EMB.nbytes / 1e6:.1f} МБ)")
print(f"выдачи     -> {ARTIFACTS / 'cascade.json'} (порядок И скоры обеих ступеней)")
print(f"замеры     -> {ARTIFACTS / 'run-cascade.json'} ({len(RUN)} ключей)")
print("на неделе 9 hw-alliance добавит третью ступень к этим двум")

**Что видно.** Три артефакта на диске, и самый тяжёлый из них — матрица эмбеддингов. Сравнивать
надо не размеры файлов, а **что переиспользуемо, а что нет**: эмбеддинги стоили минуту счёта
и не изменятся, выдачи стоили минуты и зависят от конфигурации, а `run.json` невесом и хранит
условия, при которых всё это получено. Отдельно отметь, что в `cascade.json` лежат не только
порядки, но и **скоры** обеих ступеней. Порядок отвечает на вопрос «кто выше», скор — «насколько»,
и слить два ранжирования, имея только порядки, нельзя: придётся пересчитывать ступени заново.
Ровно это понадобится на неделе 9. Чего этот вывод НЕ показывает: совместимости с ноутбуком
недели 9 — её проверит уже он, и если чего-то не окажется, построит своё в уменьшенном масштабе.
Что делать: не удалять `artifacts`. Пересчитать можно, но это минуты счёта каждый раз, а главное —
пересчёт с другим сидом даст **другие** числа, и сравнивать их с сегодняшними будет нельзя.

<details><summary>Что именно понадобится на неделе 9 — и почему сохраняем выдачи, а не только метрики</summary>

Мы сохранили три вещи: эмбеддинги, выдачи обеих ступеней и замеры. Каждая понадобится, и по
разным причинам.

**Эмбеддинги — потому что дороги.** Минута счёта на полутора тысячах документов, часы
на реальном корпусе. Они не изменятся, пока не сменится модель, и потому это классический
кэш: ключ — имя модели плюс версия корпуса.

**Выдачи — потому что позволяют пересчитать любую метрику задним числом.** Это приём, который
я не применил на неделе 4 и назвал там долгом. Имея сохранённые списки топ-k, можно посчитать
nDCG, MAP, Success@k, RBO — что угодно — не запуская ни одной модели. Стоит килобайты,
экономит часы. На неделе 9, когда появится третья ступень, сравнивать её надо будет ровно
с этими списками.

**Замеры с конфигурацией — потому что через три недели ты не вспомнишь.** В `run.json` лежат
имена моделей, размер корпуса, число запросов, сид и все полученные числа. Без имени модели
любое сравнение «было-стало» бессмысленно: половина эффекта может объясняться тем, что версия
`sentence-transformers` подтянула другой чекпойнт.

**Скоры сохранены вместе с порядками** — и это не педантизм. Слияние двух ранжирований
(неделя 9) складывает **скоры**, а не позиции: из порядка нельзя узнать, обошёл документ
соседа на волос или на пропасть. Имея только порядки, пришлось бы пересчитывать обе ступени
заново — минуты счёта каждый раз и, что хуже, риск получить чуть другие числа из-за другой
версии модели.

**Единственный честный способ слить одни порядки — RRF** (reciprocal rank fusion): складывать
`1/(k + rank)` вместо скоров. Он работает на удивление хорошо и именно потому популярен,
что не требует сопоставимых шкал. Но это выбор, а не необходимость, и делать его надо осознанно,
а не потому, что скоры не сохранили.

**Что сделает неделя 9.** Добавит третью ступень — гибрид лексического и плотного скоринга
с обучаемым слиянием, — и главный вопрос будет тем же, что сегодня: отличим ли выигрыш от шума.
С сохранёнными выдачами ответ на него займёт минуты вместо часа пересчёта.
</details>

<details><summary>Ограничения этого семинара, которые надо назвать вслух</summary>

Полный список того, где мы срезали угол, и в какую сторону это смещает выводы.

**Псевдозапросы.** Лексически щедры к BM25 и не похожи на пользовательские. Смещение: занижает
преимущество нейронных методов, то есть работает **против** вывода, который мы хотели бы
получить. Это единственное смещение сегодня, которое смещает не в нашу пользу, и потому самое
безопасное.

**Один релевантный документ.** Делает MRR дискретной и очень шумной. Смещение: расширяет
доверительные интервалы, то есть толкает к выводу «не отличимо». Значительная часть сегодняшней
неразличимости — отсюда.

**Восемьдесят запросов.** Мало. При таком разбросе различимы только эффекты порядка 0,08–0,10,
а мы искали 0,04. Замер не обладал разрешением с самого начала, и это надо было понимать
до прогона, а не после.

**Корпус в полторы тысячи документов.** На три порядка меньше любого прода. Все выводы
о **задержке** качественные, о полноте — оптимистичные: найти нужное среди полутора тысяч
проще, чем среди миллиона.

**Замер задержки на CPU без прогрева.** Смещение: завышает цену малых глубин (батч
недозагружен) и потому **преуменьшает** выгоду от выбора малой глубины.

**Модели не дообучались на нашем домене.** Обе взяты как есть, обученными на общем домене
и на MS MARCO. Это типичная ситуация для старта проекта и нетипичная для зрелого — там
би-энкодер почти всегда дообучают, и об этом лекция L11 про негативы.

<summary>Как сделать правильно, если есть бюджет</summary>
Сгенерировать триста запросов вместо восьмидесяти, взять по три релевантных документа на запрос
через кластеризацию, мерить nDCG@10 вместо MRR и повторить прогон. Это часа три счёта на CPU
и полностью снимает вопрос о разрешении.
</details>

---

## Решения

**Подглядеть — не поражение. Поражение — уйти с занятия, не поняв, где был затык.**

<details><summary>Задание 1 · рабочая глубина</summary>

```python
max_mrr = max(s["mrr"] for s in sweep.values())
best_depth = next(d for d in sorted(DEPTHS) if max_mrr - sweep[d]["mrr"] < 0.01)
overpay = sweep[max(DEPTHS)]["ms"] / sweep[best_depth]["ms"]
```

Условие, при котором за глубину стоит платить: **потолок ещё не выбран, а доля реализованного
не падает.** Проверяется это ровно тем прогоном, который мы и сделали: если при удвоении глубины
MRR растёт заметнее, чем на порог различимости, глубину стоит удваивать дальше. Если растёт
только потолок — не стоит.

Обрати внимание на порог 0,01. Он взят не с потолка: это примерно половина ширины
доверительного интервала из части 4, то есть заведомо меньше разрешения нашего замера. Порог
меньше разрешения делал бы выбор глубины подгонкой под шум.
</details>

<details><summary>Задание 2 · кого чинить</summary>

```python
realized = {d: sweep[d]["mrr"] / ceilings[d] for d in DEPTHS}
fix_stage = "reranker" if realized[max(DEPTHS)] < realized[min(DEPTHS)] else "retriever"
```

Наблюдение, которое перевернуло бы решение: если бы доля реализованного была близка к единице
на всех глубинах, узким местом однозначно был бы ретривер — судья выжимал бы из кандидатов
всё возможное, и единственный способ расти состоял бы в том, чтобы приносить ему более полный
набор. У нас доля около двух третей и падает с глубиной, то есть судья уже не справляется
с тем, что ему дают, и наращивать поток бессмысленно.

Практический вывод общего вида: **сначала смотри на потолок, потом на долю его реализации.**
Первое число говорит, есть ли что улучшать, второе — кто мешает.
</details>

<details><summary>Задание 3 · почему полнота</summary>

**Первый пункт.** Потолок качества всего каскада равен `Recall@d` первой ступени: документ,
не попавший в кандидаты, не может быть поставлен первым никаким переупорядочиванием. Значит,
MRR первой ступени влияет на итог только через то, попал документ в топ-`d` или нет, — а сам
порядок внутри кандидатов будет полностью переписан второй ступенью. Мерить первую ступень
точностью — значит мерить ту часть её работы, которую следующая ступень выбросит.

**Второй пункт, пример.** Если второй ступени нет вовсе — то есть выдача первой ступени
показывается пользователю напрямую, — точность становится единственным, что важно, а Recall@100
не значит ничего: пользователь видит три ссылки. То же самое, если ступень вторая существует,
но применяется не всегда (например, только при высоком времени ожидания или только для платных
пользователей): тогда первая ступень одновременно и отборщик, и финальный ранкер,
и мерить её надо обеими метриками, зная, какая доля трафика идёт каким путём.

Общий вид переворота: метрика ступени определяется тем, **что происходит с её выдачей дальше**,
а не тем, что это за модель.
</details>

---

## Литература

* **Nogueira & Cho (2019), «Passage Re-ranking with BERT»** — начало массового переранжирования
  кросс-энкодерами. Короткая; там же аккуратно показано, что глубина имеет предел полезности.
* **Karpukhin et al. (2020), «Dense Passage Retrieval»** — канонический би-энкодер. Читать ради
  раздела про отрицательные примеры: он объясняет, почему первая ступень учится полноте.
* **Reimers & Gurevych (2019), «Sentence-BERT»** — откуда взялась архитектура, которой мы
  кодировали корпус, и почему сырой `[CLS]` не годится.
* **Lee, Chang & Toutanova (2019), «Latent Retrieval for Weakly Supervised Open Domain QA»** —
  inverse cloze task, то есть приём, которым мы построили запросы.
* **Thakur et al. (2021), «BEIR»** — бенчмарк переносимости. Главный урок для сегодняшнего
  занятия: модель, обученная на MS MARCO, на чужой коллекции часто проигрывает BM25. Ровно то,
  что мы наблюдали.
* **Лекция L10 «Разведчики и судьи»** и `data/l7-*.json` — числа, с которыми мы сверялись.

**Дальше по курсу.** L11 объясняет, откуда берутся трудные негативы, на которых учится
би-энкодер, — и почему без них он остаётся посредственным. L12 добавит late interaction
и разреженные представления. Неделя 9 соберёт три ступени в одно ранжирование и снова спросит,
отличим ли выигрыш от шума.

## Дамп прогона

Правило 10.5: занятие не считается прогнанным, пока его числа не лежат в файле рядом
с конфигурацией рантайма. Ячейка ниже собирает все численные результаты ноутбука —
от сида до финальных метрик — и кладёт их в `runs/lab-cascade.json`. Это и есть
доказательство прогона: разбор сверяется с файлом, а не с памятью автора.

In [ ]:
# Дамп прогона — все числовые результаты + конфигурация рантайма (правило 10.5).
import json as _json, os as _os, sys as _sys, platform as _pl, pathlib as _pathlib

_runtime = {"python": _sys.version.split()[0], "platform": _pl.platform()}
_torch = _sys.modules.get("torch")   # НЕ импортируем сами: рамка 7.4 — сид и пин
if _torch is not None:                # обязателен только там, где ноутбук torch ИСПОЛЬЗУЕТ
    _runtime["torch"] = _torch.__version__
    _runtime["gpu"] = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else None
else:
    import subprocess as _sp
    try:
        _q = _sp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True, timeout=5)
        _runtime["gpu"] = (_q.stdout.strip().splitlines() or [None])[0] if _q.returncode == 0 else None
    except Exception:
        _runtime["gpu"] = None

def _plain(v):
    try:
        import numpy as _np
        if isinstance(v, _np.integer): return int(v)
        if isinstance(v, _np.floating): return float(v)
    except Exception:
        pass
    return v

def _num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)

_metrics = {}
for _k, _v in sorted(globals().items()):
    if _k.startswith("_") or (len(_k) == 1 and _k.islower()):
        continue                       # служебные имена и счётчики циклов
    _v = _plain(_v)
    if _num(_v):
        _metrics[_k] = _v
    elif isinstance(_v, dict) and 0 < len(_v) <= 64 and all(_num(_plain(_x)) for _x in _v.values()):
        _metrics[_k] = {str(_kk): _plain(_vv) for _kk, _vv in _v.items()}
    elif isinstance(_v, (list, tuple)) and 0 < len(_v) <= 64 and all(_num(_plain(_x)) for _x in _v):
        _metrics[_k] = [_plain(_x) for _x in _v]

_out = _pathlib.Path(_os.environ.get("RUNS_DIR", "runs")); _out.mkdir(parents=True, exist_ok=True)
_path = _out / "lab-cascade.json"
_json.dump({"notebook": "lab-cascade", "runtime": _runtime, "metrics": _metrics},
           open(_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1, sort_keys=True)
print(f"дамп: {_path} · величин: {len(_metrics)} · рантайм: {_runtime['gpu'] or 'CPU'}")

**Что видно.** В дампе — конфигурация прогона и все скалярные результаты по именам
переменных. Сравнивать надо не тайминги — они свойство рантайма, и на T4, A100 и CPU
законно разные, — а метрики качества: при одном сиде они обязаны совпасть до знака.
Если твой прогон разошёлся с эталонным в качестве, а не во времени, — это находка,
неси её на занятие.